In [1]:
import pandas as pd
from vrpy import VehicleRoutingProblem
import networkx as nx
from tqdm import tqdm 
import json
import warnings
import logging
import time
import numpy as np

warnings.filterwarnings("ignore")
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)


In [2]:
df_quadrados = pd.read_csv('C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Utils\\quadrados_clusterizados_todos_os_clusters.csv').drop('Unnamed: 0', axis=1).drop(['maxLat','minLat','maxLong','minLong'], axis=1)
df_quadrados['Quadrado'] = df_quadrados['Quadrado'].apply(int)
df_quadrados = df_quadrados.astype({'candidate':'string', 'Quadrado': 'string'})

df_demandas = pd.read_csv('C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Utils\\df_demanda_quadrados.csv').drop('Unnamed: 0', axis=1)
df_demandas['Quadrado'] = df_demandas['Quadrado'].apply(int)
df_demandas = df_demandas.astype({'Data':'string', 'Quadrado': 'string'})

df_distancias_cds = pd.read_csv('C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Utils\\df_distancias_cds.csv').drop('Unnamed: 0', axis=1)
df_distancias_cds['quadrado'] = df_distancias_cds['quadrado'].apply(int)
df_distancias_cds = df_distancias_cds.astype({'quadrado': 'string', 'CD': 'string'})

df_distancias_quadrados = pd.read_csv('C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Utils\\df_distancias_quadrados.csv').drop('Unnamed: 0', axis=1)
df_distancias_quadrados['quadrado1'] = df_distancias_quadrados['quadrado1'].apply(int)
df_distancias_quadrados['quadrado2'] = df_distancias_quadrados['quadrado2'].apply(int)
df_distancias_quadrados = df_distancias_quadrados.astype({'quadrado1': 'string', 'quadrado2': 'string'})

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Utils\\quadrados_clusterizados_todos_os_clusters.csv'

In [3]:
# define as variaveis de iteração
num = df_quadrados['num'].sort_values(ascending=False).unique()
instancias = df_demandas.drop(['Quadrado', 'Data'], axis=1).columns
dias = df_demandas['Data'].unique()

# define as variáveis de restrição
speed = {'carro': 32, 'moto': 40}
loading_time = 0.08
cost_per_km = {'carro': 1.5, 'moto': 1}
load_capacity=60 
duration=6
fixed_cost=80

In [4]:
def addResult(n, i, d, c, prob):
    
    result = {}
    result['num'] = [n]
    result['instancia'] = [i]
    result['dia'] = [d]
    result['cluster'] = [c]
    result['numero de rotas'] = len(prob.best_routes)
    result['tempo médio por rota'] = sum(list(prob.best_routes_duration.values()))/len(prob.best_routes)
    result['custo total das rotas'] = sum(list(prob.best_routes_cost.values()))
    result['custo maximo de rota'] = max(list(prob.best_routes_cost.values()))
    
    j=0
    custoTotal = 0
    countTotal = 0
    vetor = []
    for rota in list(prob.best_routes.values()):
        count = len(rota) - 2
        custo = list(prob.best_routes_cost.values())[j]
        custoTotal = custoTotal + custo
        countTotal = countTotal + count
        vetor.append(custo/count)
        j = j+1

    result['frete medio'] = sum(vetor)/len(vetor)
    result['frete maximo'] = max(vetor)
    result['frete minimo'] = min(vetor)
    result['clientes'] = countTotal

    return result

In [ ]:
resultado_final = pd.DataFrame(columns=['num', 'instancia', 'dia', 'cluster', 'numero de rotas', 'custo total das rotas', 'custo maximo de rota', 
                                        'frete medio', 'clientes', 'frete maximo', 'frete minimo', 'tempo médio por rota'])

# itera pelo número de clusters (1, 3, 5 e 7)
for n in num:
    if n == 1:
        break
    quadrados = df_quadrados[df_quadrados['num'] == n]

    print(n)
    # itera por cada instância
    for i in instancias:
        demandas = df_demandas[['Quadrado', 'Data', str(i)]]
        
        group = demandas[['Data', str(i)]].groupby(by='Data').sum().reset_index()
        sort = group.sort_values(by=str(i), ascending=False).reset_index()
        dataMax = sort['Data'][0:30]

        # itera por cada dia
        for d in dataMax:
            # define o data frame principal com as demandas por cada quadrado do cluster da iteração
            df_demandas_cluster = demandas[demandas['Data'] == str(d)].merge(quadrados, on='Quadrado', how='left')
            df_demandas_cluster = df_demandas_cluster[df_demandas_cluster[i] > 0]
            
            # itera por cada cluster
            clusters = df_demandas_cluster['cluster'].unique()
            for c in clusters:
                # define os dados
                # data frame principal com as demandas por cada quadrado para o cluster específico da iteração
                df = df_demandas_cluster.loc[df_demandas_cluster['cluster'] == c].reset_index().drop(['index'], axis=1)
                
                # data frame com as distancias entre os quadrados e os CDs 
                distancias_cds = df_distancias_cds[(df_distancias_cds['CD'].isin(df['candidate']))]
                distancias_cds = distancias_cds[(distancias_cds['quadrado'].isin(df['Quadrado']))]
                
                # data frame com as distancias entre os quadrados
                distancias_quadrados = df_distancias_quadrados[(df_distancias_quadrados['quadrado1'].isin(df['Quadrado']))]
                distancias_quadrados = distancias_quadrados[(distancias_quadrados['quadrado2'].isin(df['Quadrado']))]

                # converte a distancia em custo e tempo
                df_custo = distancias_quadrados
                df_custo['Custo'] = df_custo['Distance'] * cost_per_km['carro']
                df_custo['Tempo'] = df_custo['Distance']/speed['carro'] + loading_time

                # converte a coluna em dicionário
                df_custo['attributes'] = '{"cost": ' + df_custo['Custo'].apply(str) + ', "time": ' + df_custo['Tempo'].apply(str) + '}'
                df_custo['attributes'] = df_custo['attributes'].apply(json.loads)
                
                # declara o grafo e cria os nós
                G = nx.from_pandas_edgelist(df_custo, 'quadrado1', 'quadrado2', create_using=nx.DiGraph())

                # cria os vértices e adiciona os custos a partir da matriz de custos (df_custos)
                G.add_edges_from(list(df_custo[['quadrado1', 'quadrado2', 'attributes']].itertuples(index=False, name=None)))
                
                # cria os vértices e custos incluindo a Source e Sink (CD do cluster)
                for k, linha in distancias_cds.iterrows():
                    G.add_edge("Source", linha['quadrado'],cost=linha['Distance'])
                    G.add_edge(linha['quadrado'], "Sink",cost=linha['Distance'])
                    
                # define a demanda para cada cliente
                for o, r in df.iterrows():
                    nx.set_node_attributes(G, {r['Quadrado']: r[str(i)]}, 'demand')

                # resolve o problema utilizando o grafo
                prob = VehicleRoutingProblem(G, mixed_fleet=False, load_capacity=load_capacity, duration=duration, fixed_cost=fixed_cost)
                prob.solve(time_limit=10, heuristic_only=True, cspy=False)
                
                # salva os resultados resultados
                result = addResult(n, i, d, c, prob)
                resultado_final = resultado_final.append(pd.DataFrame(result)) 
                
resultado_final = resultado_final.reset_index().drop(columns='index', index=1)
resultado_final['custo fixo'] = 250 * resultado_final['num']
resultado_final['custo total'] = resultado_final['custo fixo'] + resultado_final['custo total das rotas']
resultado_final['custo medio por rota'] = resultado_final['custo total']/resultado_final['numero de rotas']
resultado_final['dispersao max de custo'] = resultado_final['custo maximo de rota'] - resultado_final['tempo médio por rota']

In [10]:
resultado = resultado_final

In [11]:
resultado.to_csv('C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Resultados\\resultado_final.csv')